# Granger Causality vs Functional Connectivity

## Core Methodological Argument

This notebook demonstrates why Functional Connectivity (FC, Pearson correlation) **cannot** recover directed effective connectivity, and why Granger causality is necessary for validating directed edges.

### Key Points:
1. FC is symmetric by construction (FC = FC.T)
2. VAR(1) A matrix is asymmetric (captures directed relationships)
3. FC symmetry forces real eigenvalues - cannot encode oscillatory dynamics
4. Spurious correlations exist: high FC without causal edge
5. Hidden causal edges exist: significant Granger with low FC

### Conclusion:
Spectral inversion on symmetric FC cannot recover the asymmetric A matrix required for Network Control Theory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'src'))
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..'))

np.random.seed(42)

print("Imports successful")

---
## Section 1: Generate Synthetic VAR(1) Data

We generate synthetic data from a known stable VAR(1) system:
```
X[t] = A_true @ X[t-1] + noise
```

This gives us ground truth to compare against.

In [ ]:
def generate_stable_A(N=10, rho=0.7, seed=None):
    """Generate random stable asymmetric connectivity matrix."""
    if seed is not None:
        np.random.seed(seed)
    
    A = np.random.randn(N, N) * 0.3
    np.fill_diagonal(A, 0)  # no self-loops
    
    # Scale to target spectral radius
    eigvals = np.linalg.eigvals(A)
    current_rho = np.max(np.abs(eigvals))
    if current_rho > 1e-8:
        A = A * (rho / current_rho)
    
    return A


def generate_var1_timeseries(A, T=300, seed=None):
    """Generate VAR(1) timeseries: X[t] = A @ X[t-1] + noise."""
    if seed is not None:
        np.random.seed(seed)
    
    N = A.shape[0]
    X = np.zeros((T, N))
    X[0] = np.random.randn(N) * 0.1
    
    for t in range(1, T):
        X[t] = A @ X[t-1] + np.random.randn(N) * 0.1
    
    return X


# Generate synthetic data
N = 10
T = 300
rho = 0.7

A_true = generate_stable_A(N=N, rho=rho, seed=42)
X = generate_var1_timeseries(A_true, T=T, seed=42)

print(f"Generated synthetic VAR(1) data:")
print(f"  N = {N} nodes")
print(f"  T = {T} timepoints")
print(f"  rho = {np.max(np.abs(np.linalg.eigvals(A_true))):.4f}")
print(f"  A_true asymmetry: mean|A - A.T| = {np.mean(np.abs(A_true - A_true.T)):.4f}")

---
## Section 2: Compute Functional Connectivity (FC)

FC = Pearson correlation matrix. By construction, FC is symmetric.

In [ ]:
# Compute FC
FC = np.corrcoef(X.T)
np.fill_diagonal(FC, 0)  # no self-correlation for comparison

# Verify symmetry
fc_symmetry_error = np.max(np.abs(FC - FC.T))
fc_asymmetry = np.mean(np.abs(FC - FC.T))

print(f"Functional Connectivity (Pearson correlation):")
print(f"  Shape: {FC.shape}")
print(f"  Max |FC - FC.T|: {fc_symmetry_error:.2e} (should be 0)")
print(f"  Mean |FC - FC.T|: {fc_asymmetry:.2e} (should be ~0)")
print(f"  FC is symmetric: {np.allclose(FC, FC.T)}")

# Visualize FC
plt.figure(figsize=(6, 5))
plt.imshow(FC, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.title('Functional Connectivity (Symmetric)')
plt.xlabel('ROI')
plt.ylabel('ROI')
plt.tight_layout()
plt.savefig('figures/fc_matrix.png', dpi=150)
plt.show()
print("\nFC matrix saved to figures/fc_matrix.png")

---
## Section 3: Compute VAR(1) Effective Connectivity (A)

VAR(1) OLS estimation recovers the directed connectivity matrix A.

```
X[1:] = X[:-1] @ A.T + noise
A.T = lstsq(X[:-1], X[1:])
```

In [ ]:
# Fit VAR(1) via OLS
X_past = X[:-1]
X_future = X[1:]

A_est_T, _, _, _ = np.linalg.lstsq(X_past, X_future, rcond=None)
A_var1 = A_est_T.T
np.fill_diagonal(A_var1, 0)  # no self-loops

# Check asymmetry
var1_asymmetry = np.mean(np.abs(A_var1 - A_var1.T))
recovery_error = np.mean(np.abs(A_var1 - A_true))

print(f"VAR(1) OLS Effective Connectivity:")
print(f"  Shape: {A_var1.shape}")
print(f"  Mean |A - A.T|: {var1_asymmetry:.4f} (should be > 0)")
print(f"  Recovery error |A_est - A_true|: {recovery_error:.4f}")
print(f"  A is asymmetric: {not np.allclose(A_var1, A_var1.T)}")

# Visualize A_var1
plt.figure(figsize=(6, 5))
plt.imshow(A_var1, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
plt.colorbar(label='Connection Strength')
plt.title('VAR(1) Effective Connectivity (Asymmetric)')
plt.xlabel('ROI (cause)')
plt.ylabel('ROI (effect)')
plt.tight_layout()
plt.savefig('figures/var1_ec_matrix.png', dpi=150)
plt.show()
print("\nVAR(1) EC matrix saved to figures/var1_ec_matrix.png")

---
## Section 4: Granger Causality - Spurious and Hidden Edges

Granger causality tests whether past values of node j improve prediction of node i.

- **Spurious correlation**: FC[i,j] > 0.3 but G[i,j] = 0 (high correlation, no causation)
- **Hidden causal edge**: G[i,j] = 1 but |FC[i,j]| < 0.1 (causation without correlation)

In [ ]:
from neurosim.connectivity.granger import granger_causality_matrix, causality_vs_correlation_summary

# Run causality vs correlation summary
summary = causality_vs_correlation_summary(A_var1, X, order=1, alpha=0.05)

print("Causality vs Correlation Summary:")
print(f"  FC asymmetry: {summary['fc_asymmetry']:.2e} (confirming FC is symmetric)")
print(f"  VAR(1) asymmetry: {summary['var1_asymmetry']:.4f} (confirming A is asymmetric)")
print(f"  Spurious correlations (FC>0.3, G=0): {summary['n_spurious']}")
print(f"  Hidden causal edges (G=1, |FC|<0.1): {summary['n_hidden']}")

G = summary['G']
FC = summary['FC']

# Visualize Granger causality matrix
plt.figure(figsize=(6, 5))
plt.imshow(G, cmap='Greys', vmin=0, vmax=1)
plt.colorbar(label='Causal (0/1)')
plt.title('Granger Causality Matrix (Asymmetric)')
plt.xlabel('ROI (cause)')
plt.ylabel('ROI (effect)')
plt.tight_layout()
plt.savefig('figures/granger_causality_matrix.png', dpi=150)
plt.show()
print("\nGranger causality matrix saved to figures/granger_causality_matrix.png")

---
## Section 5: Eigenvalue Spectrum Analysis

### Key Insight: FC Symmetry Forces Real Eigenvalues

By the spectral theorem, symmetric matrices have purely real eigenvalues.

Asymmetric A from VAR(1) has complex eigenvalues encoding oscillatory dynamics.

Feeding symmetric FC into NCT collapses complex eigenvalues, destroying causal geometry.

In [ ]:
# Compute eigenvalues
eigvals_FC = np.linalg.eigvals(FC)
eigvals_A = np.linalg.eigvals(A_var1)

# Check for complex eigenvalues
max_imag_FC = np.max(np.abs(eigvals_FC.imag))
max_imag_A = np.max(np.abs(eigvals_A.imag))

print("Eigenvalue Analysis:")
print(f"  FC eigenvalues:")
print(f"    Max |Im(lambda)|: {max_imag_FC:.2e} (should be 0 - symmetric matrix)")
print(f"    All real: {max_imag_FC < 1e-10}")
print(f"  VAR(1) A eigenvalues:")
print(f"    Max |Im(lambda)|: {max_imag_A:.4f} (should be > 0 - asymmetric matrix)")
print(f"    Has complex pairs: {max_imag_A > 1e-6}")

# Count complex conjugate pairs in A
n_complex_A = np.sum(np.abs(eigvals_A.imag) > 1e-6)
print(f"  VAR(1) A has {n_complex_A} complex eigenvalues (out of {N})")

In [ ]:
# Plot eigenvalue spectra
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# FC eigenvalue spectrum (all real)
ax1.scatter(eigvals_FC.real, eigvals_FC.imag, c='blue', alpha=0.7, s=50, label='FC eigenvalues')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Real')
ax1.set_ylabel('Imaginary')
ax1.set_title('FC Eigenvalue Spectrum\n(All Real - Symmetric Matrix)')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
ax1.legend()

# VAR(1) A eigenvalue spectrum (complex pairs)
ax2.scatter(eigvals_A.real, eigvals_A.imag, c='red', alpha=0.7, s=50, label='A eigenvalues')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Real')
ax2.set_ylabel('Imaginary')
ax2.set_title('VAR(1) A Eigenvalue Spectrum\n(Complex Pairs - Asymmetric Matrix)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('figures/eigenvalue_spectrum_comparison.png', dpi=150)
plt.show()

print("\nEigenvalue spectrum comparison saved to figures/eigenvalue_spectrum_comparison.png")
print("\n*** FC symmetry forces real eigenvalues - cannot encode directed dynamics ***")

---
## Section 6: Conclusion

### Why spectral_inversion_solver(corrcoef(...)) Cannot Recover Directed A

1. **Symmetry constraint**: FC = FC.T by construction of Pearson correlation
   - This means FC has N*(N-1)/2 independent parameters (upper triangle)
   - Directed A has N*(N-1) independent parameters (full matrix minus diagonal)
   - Information loss: 50% of directional information is discarded

2. **Eigenvalue collapse**: Symmetric matrices have purely real eigenvalues
   - Complex eigenvalues encode oscillatory/rotational dynamics
   - These are essential for NCT metrics (modal controllability depends on eigenvectors)
   - FC cannot capture these dynamics

3. **Spurious correlations**: High FC does not imply causation
   - Common drivers create correlation without direct connection
   - Indirect paths (A->B->C) create FC[A,C] without A->C edge

4. **Hidden edges**: Significant Granger causality with low FC
   - Directed relationships can exist without correlation
   - These are invisible to FC-based methods

### The Methodological Imperative

**Network Control Theory requires directed, asymmetric connectivity matrices.**

Using symmetric FC as input to NCT:
- Produces incorrect controllability metrics
- Misses causal relationships
- Cannot explain oscillatory brain dynamics

**VAR(1) OLS estimation is the correct approach for NCT applications.**

In [ ]:
print("\n" + "=" * 70)
print("SUMMARY: FC Symmetry vs VAR(1) Asymmetry")
print("=" * 70)
print(f"  FC is symmetric: {np.allclose(FC, FC.T)}")
print(f"  FC eigenvalues are all real: {max_imag_FC < 1e-10}")
print(f"  VAR(1) A is asymmetric: {not np.allclose(A_var1, A_var1.T)}")
print(f"  VAR(1) A has complex eigenvalues: {max_imag_A > 1e-6}")
print(f"  Spurious correlations found: {summary['n_spurious']}")
print(f"  Hidden causal edges found: {summary['n_hidden']}")
print("\n  => spectral_inversion_solver(corrcoef(...)) CANNOT recover directed A")
print("  => VAR(1) OLS estimation is required for valid NCT analysis")
print("=" * 70)